# RAG Fundamentals with Gemini

This notebook demonstrates the core RAG pipeline:
1. Chunking
2. Embeddings
3. Retrieval
4. Citation-grounded answer generation

In [ ]:
import os
import re
import numpy as np
from google import genai

GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=GOOGLE_API_KEY)

GEN_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "models/gemini-embedding-001"  #  embedding model name

print("Client ready.")

## 1) Build a tiny knowledge base + chunking

In production, these would come from PDFs, docs, wikis, etc. Here we use small in-memory docs for clarity.

In [ ]:
documents = [
    {
        "source": "policy_handbook.md",
        "text": "Employees can work remotely up to three days per week. Security training is mandatory every quarter. VPN is required for accessing internal systems."
    },
    {
        "source": "it_support_faq.md",
        "text": "Password resets are available through the self-service portal. MFA enrollment is required for all accounts. For urgent incidents, contact the IT hotline."
    },
    {
        "source": "benefits_guide.md",
        "text": "Health insurance coverage starts on the first day of the month after joining. Annual wellness reimbursement is up to 300 USD. Paid parental leave is available for eligible employees."
    }
]

def chunk_text(text, chunk_size=90, overlap=20):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks

chunk_records = []
counter = 1
for doc in documents:
    for piece in chunk_text(doc["text"], chunk_size=90, overlap=20):
        cid = f"C{counter}"
        chunk_records.append({"chunk_id": cid, "source": doc["source"], "text": piece})
        counter += 1

print(f"Created {len(chunk_records)} chunks.")
for c in chunk_records:
    print(f"[{c['chunk_id']}] ({c['source']}) -> {c['text']}")

In [ ]:
# Let's check what models are available
print("Available embedding models:")
for model in client.models.list():
    if 'embed' in model.name.lower():
        print(f"  - {model.name}")

In [ ]:
def embed_texts(texts, model=EMBED_MODEL):
    """Generate embeddings for a list of texts using Gemini embedding model"""
    result = client.models.embed_content(
        model=model,
        contents=texts
    )
    
    # Extract embeddings from response
    embeddings = []
    if hasattr(result, 'embeddings'):
        for emb in result.embeddings:
            if hasattr(emb, 'values'):
                embeddings.append(np.array(emb.values, dtype=np.float32))
    
    return embeddings

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors (0=unrelated, 1=identical)"""
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

chunk_texts = [c["text"] for c in chunk_records]
chunk_vectors = embed_texts(chunk_texts)

def retrieve(query, top_k=3):
    q_vec = embed_texts([query])[0]
    scored = []
    for rec, vec in zip(chunk_records, chunk_vectors):
        score = cosine_similarity(q_vec, vec)
        scored.append((score, rec))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

query = "What are the rules for remote work and secure access?"
top_hits = retrieve(query, top_k=3)

print("Top retrieved chunks:")
for score, rec in top_hits:
    print(f"score={score:.4f} | [{rec['chunk_id']}] ({rec['source']}) {rec['text']}")

In [ ]:
def answer_with_citations(query, top_k=3):
    hits = retrieve(query, top_k=top_k)

    context_lines = []
    for _, rec in hits:
        context_lines.append(f"[{rec['chunk_id']}] source={rec['source']}\n{rec['text']}")
    context_block = "\n\n".join(context_lines)

    prompt = f"""You are a grounded assistant.
Answer ONLY using the context below.
If the context is insufficient, say: 'I don't have enough context.'
For every factual claim, include at least one citation tag like [C1].

Question:
{query}

Context:
{context_block}
"""

    resp = client.models.generate_content(model=GEN_MODEL, contents=prompt)
    answer = resp.text
    found_citations = sorted(set(re.findall(r"\[C\d+\]", answer)))

    return answer, hits, found_citations

final_answer, final_hits, cites = answer_with_citations(
    "How often is security training required and what is needed for internal system access?",
    top_k=3
)

print("Answer:")
print(final_answer)
print("\nCitations found in answer:", cites)

print("\nRetrieved evidence:")
for score, rec in final_hits:
    print(f"- [{rec['chunk_id']}] ({rec['source']}) score={score:.4f}")

## RAG with Text File from Current Folder

This demonstration shows how to load a text file, chunk it, create embeddings, and use RAG to answer questions based on the file content.

In [ ]:
# Step 1: List available text files in the current directory
import glob

current_dir = os.getcwd()
print(f"Current directory: {current_dir}\n")

# Find all .txt files
txt_files = glob.glob("*.txt")

if txt_files:
    print(f"✅ Found {len(txt_files)} text file(s):")
    for i, file in enumerate(txt_files, 1):
        file_size = os.path.getsize(file)
        print(f"  {i}. {file} ({file_size:,} bytes)")
    print(f"\n📌 Will use: {txt_files[0]} for RAG demonstration")
else:
    print("❌ No .txt files found in current directory")
    print("💡 Please add a .txt file to the current directory and run this cell again.")

In [ ]:
# Step 2: Load and chunk the text file
# Select the first text file (or modify to select specific file)
selected_file = txt_files[0]
print(f"📖 Loading file: {selected_file}\n")

# Read the file content
with open(selected_file, 'r', encoding='utf-8') as f:
    file_content = f.read()

print(f"File size: {len(file_content)} characters")
print(f"Preview (first 200 chars):\n{file_content[:200]}...\n")

# Chunk the file content with larger chunks for better context
def chunk_text_file(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks"""
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks

# Create chunks from file
file_chunks = chunk_text_file(file_content, chunk_size=300, overlap=50)

# Create chunk records with metadata
file_chunk_records = []
for idx, chunk_text in enumerate(file_chunks, 1):
    file_chunk_records.append({
        "chunk_id": f"F{idx}",
        "source": selected_file,
        "text": chunk_text.strip()
    })

print(f"✅ Created {len(file_chunk_records)} chunks from the file\n")
print("Sample chunks:")
for i, rec in enumerate(file_chunk_records[:3], 1):
    print(f"\n[{rec['chunk_id']}] ({rec['source']})")
    print(f"  Text: {rec['text'][:100]}...")

In [ ]:
# Step 3: Create embeddings for all file chunks
print("🔄 Creating embeddings for file chunks...")

file_texts = [rec["text"] for rec in file_chunk_records]
file_vectors = embed_texts(file_texts)

print(f"✅ Created {len(file_vectors)} embeddings")
print(f"📊 Embedding dimension: {len(file_vectors[0])}")
print("🎯 Ready for semantic search!")

In [ ]:
# Step 4: Create a retrieval function for the text file
def retrieve_from_file(query, top_k=3):
    """Retrieve most relevant chunks from the file based on query"""
    # Get query embedding
    query_vector = embed_texts([query])[0]
    
    # Calculate similarity scores
    scored_chunks = []
    for rec, vec in zip(file_chunk_records, file_vectors):
        score = cosine_similarity(query_vector, vec)
        scored_chunks.append((score, rec))
    
    # Sort by score and return top_k
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    return scored_chunks[:top_k]

# Test the retrieval with a sample query
test_query = "How to apply for a loan?"
print(f"🔍 Query: {test_query}\n")

results = retrieve_from_file(test_query, top_k=3)

print("📋 Top 3 relevant chunks:\n")
for rank, (score, rec) in enumerate(results, 1):
    print(f"{rank}. Score: {score:.4f} | [{rec['chunk_id']}]")
    print(f"   Text: {rec['text'][:150]}...")
    print()

In [ ]:
# Step 5: Implement RAG - Retrieval-Augmented Generation
def rag_answer_from_file(query, top_k=3):
    """
    RAG pipeline: 
    1. Retrieve relevant chunks from file
    2. Build context from retrieved chunks
    3. Generate answer using LLM with context
    """
    # Retrieve relevant chunks
    results = retrieve_from_file(query, top_k=top_k)
    
    # Build context block
    context_parts = []
    for score, rec in results:
        context_parts.append(f"[{rec['chunk_id']}] (relevance: {score:.2f})\n{rec['text']}")
    
    context_block = "\n\n".join(context_parts)
    
    # Create prompt for LLM
    prompt = f"""You are a helpful assistant that answers questions based ONLY on the provided context.

Instructions:
- Answer the question using ONLY information from the context below
- If the context doesn't contain enough information, say "I cannot answer based on the provided context"
- Include citations like [F1], [F2] to reference specific chunks
- Be concise and accurate

Question: {query}

Context from file:
{context_block}

Answer:"""
    
    # Generate answer using Gemini
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )
    
    answer = response.text
    
    # Extract citations used
    citations = sorted(set(re.findall(r"\[F\d+\]", answer)))
    
    return {
        "answer": answer,
        "query": query,
        "retrieved_chunks": results,
        "citations": citations
    }

print("✅ RAG function ready!")

In [ ]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

# List of test questions
test_questions = [
    "What are the key features of Python?",
    "How is Python used in machine learning?",
    "What are some best practices for Python programming?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)
    
    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)
    
    print(f"\n📝 Answer:")
    print(result["answer"])
    
    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")
    
    print(f"\n📚 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

In [ ]:
# Step 6: Test RAG with sample questions
print("=" * 80)
print("🤖 RAG DEMONSTRATION - Question Answering from Text File")
print("=" * 80)

# List of test questions
test_questions = [
    "How to apply for a loan?",
    "Any customer care details?",
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'=' * 80}")
    print(f"Question {i}: {question}")
    print("=" * 80)
    
    # Get RAG answer
    result = rag_answer_from_file(question, top_k=3)
    
    print(f"\n📝 Answer:")
    print(result["answer"])
    
    print(f"\n🔗 Citations used: {', '.join(result['citations']) if result['citations'] else 'None'}")
    
    print(f"\n📚 Retrieved evidence:")
    for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
        print(f"  {rank}. [{rec['chunk_id']}] Relevance: {score:.4f}")
        print(f"     Preview: {rec['text'][:100]}...")

print(f"\n{'=' * 80}")
print("✅ RAG demonstration complete!")
print("=" * 80)

In [ ]:
# Step 7: Interactive RAG - Ask your own question!
# Modify the query below to ask your own questions about the file

custom_query = "Is online banking facility available?"

print("🔍 Custom Query:", custom_query)
print("=" * 80)

result = rag_answer_from_file(custom_query, top_k=3)

print(f"\n💡 Answer:\n{result['answer']}")

print(f"\n📊 Retrieval Statistics:")
print(f"   • Retrieved chunks: {len(result['retrieved_chunks'])}")
print(f"   • Citations found: {len(result['citations'])}")
print(f"   • Source file: {selected_file}")

print(f"\n📖 Relevant Context Used:")
for rank, (score, rec) in enumerate(result["retrieved_chunks"], 1):
    print(f"\n{rank}. [{rec['chunk_id']}] (Relevance: {score:.4f})")
    print(f"   {rec['text'][:200]}...")

## 📚 RAG Process Summary

**What we demonstrated:**

1. **📁 File Loading**: Automatically detected or created a text file in the current directory
2. **✂️ Chunking**: Split the file into overlapping chunks (300 chars with 50 char overlap)
3. **🧮 Embedding**: Generated vector embeddings using `models/gemini-embedding-001`
4. **🔍 Retrieval**: Found most relevant chunks using cosine similarity
5. **🤖 Generation**: Used Gemini LLM to answer questions with citations
6. **🎯 Grounding**: Ensured answers are based ONLY on the provided context

**Key Benefits of RAG:**
- ✅ Reduces hallucinations by grounding answers in source documents
- ✅ Provides citations for transparency and verification
- ✅ Enables Q&A over custom documents without fine-tuning
- ✅ Dynamically updates knowledge base (just update the file)

**To use with your own file:**
- Replace the file in Step 1 or modify `selected_file` variable
- Adjust `chunk_size` and `overlap` for your content type
- Modify `top_k` parameter to retrieve more/fewer chunks